# GAT M1·M2·M4·M5 2×2 요인실험 (Dunnhumby, seed 42)

Sparse single-head GAT 안에서 M2 표현과 M4 손실가중을 각각 끄고 켠 4개 arm을 고정 100 epoch로 새로 학습합니다. M1·M4는 67차원 ID, M2·M5는 64차원 ID + 3차원 `q_C×q_V` 가격위치 기저를 사용합니다.

`DAY 1~683` 학습, `DAY 684~690` 신규상품 개발평가이며 final test·holdout은 만들지 않습니다. CLV 배정 순열 대조군이 없으므로 귀속은 판정하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys
PINNED_SOURCE_COMMIT = '34e2ed725678b7a9cc675ff0ac3b460ed7208434'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
repo = Path('/content/clv-m2-lightgcn-runner')
errors = []
for attempt in range(1, 4):
    os.chdir('/content')
    if repo.exists(): shutil.rmtree(repo)
    result = subprocess.run(['git','clone',REPO_URL,str(repo)], text=True, capture_output=True)
    if result.returncode == 0: break
    errors.append(result.stderr.strip()); print(f'GitHub clone {attempt}/3 실패:', result.stderr.strip())
else: raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(errors))
subprocess.run(['git','-C',str(repo),'checkout','-q',PINNED_SOURCE_COMMIT], check=True)
actual_sha = subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'], text=True).strip()
assert actual_sha == PINNED_SOURCE_COMMIT
for name in tuple(sys.modules):
    if name.startswith(('gnn_clv_','clv_scaled_','lightgcn_','clv_')): del sys.modules[name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json, torch
from gnn_clv_m1_m2_m4_m5_factorial_screen import configure_gnn_clv_factorial_screen, model_ids, preflight_summary, run_gnn_clv_factorial_screen
assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_gnn_clv_factorial_screen('gat')
summary = preflight_summary(cfg)
assert summary['trained_models'] == list(model_ids('gat').values())
assert summary['fixed']['new_item_task'] and summary['fixed']['min_item_interactions'] == 1
assert not summary['fixed']['final_test_constructed'] and not summary['fixed']['holdout_constructed']
assert summary['fixed']['one_training_loop_and_optimizer_per_arm']
assert not summary['m2']['external_reranking']
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_gnn_clv_factorial_screen(cfg)

In [ ]:
from IPython.display import display
def show(frame):
    view = frame.copy(); view.attrs = {}; display(view)
core = ['model_id','role','recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50','price_purchase_amount_weighted_hit@10','vndcg@10','coverage@10']
print('1) GAT M1·M2·M4·M5 핵심 절대지표'); show(result_df[[c for c in core if c in result_df.columns]])
print('2) M1·M4 대비 전체 지표 비교'); show(result_df.attrs['comparison'])
print('3) M2·M4 요인효과와 상호작용'); show(result_df.attrs['interaction'])
print('4) 동일 학습모형의 M2 입력 on/off 진단'); show(result_df.attrs['counterfactual_m2']); show(result_df.attrs['top10_overlap'])
print('5) 사전 고정 판독'); print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('6) 저장 파일'); print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))